# Atividade Prática - Parte 1
### Grupo 2:<br>
Victor Monteiro <br>
Arthur Horta <br>
João Henrique <br>
Pedro Fioravante <br>

### Tarefa 1: Importe os dados para este notebook e Gere as Estatísticas descritivas da variável dependente e da variável explicativa

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from pathlib import Path

# ------------------------------ CONFIGURAÇÃO INICIAL ------------------------------

# Dicionário com os tickers das empresas
TICKERS = {
    "LIGHT":      "LIGT3.SA",
    "BRASKEM":    "BRKM5.SA",
    "USIMINAS":   "USIM5.SA",
    "CEMIG":      "CMIG4.SA",
    "ITAUBANCO":  "ITUB4.SA",
}

# Ticker do IBOVESPA e janela de análise (dias antes e depois da divulgação)
IBOV, WINDOW = "^BVSP", 90

# Diretórios de entrada e saída
DIR = Path("Dados")              # Onde estão os arquivos CSV com os preços
ROOT = Path("Plot")             # Onde serão salvos os gráficos e tabelas
ROOT.mkdir(exist_ok=True)      # Cria a pasta de saída se não existir

# ------------------------------ LEITURA DA LISTA DE EVENTOS ------------------------------

# Lista com as empresas e datas de divulgação de resultados
tabela = pd.read_csv(DIR / "Lista_Empresas.csv", delimiter=";")

# ------------------------------ LOOP SOBRE CADA EMPRESA ------------------------------

for _, row in tabela.iterrows():
    emp = row["Empresas"]
    ticker = TICKERS[row["Empresas"]]
    ref = datetime.strptime(row["Data de divulgação"], "%d/%m/%Y")  # Data de referência
    ini, fim = ref - timedelta(days=WINDOW), ref + timedelta(days=WINDOW)  # Janela de análise
    ano = ref.year

    # ------------------------------ FUNÇÃO DE LEITURA DOS PREÇOS ------------------------------
    def load(tk):
        """
        Lê os dados de preços de fechamento ajustado ('Close') de um ticker específico dentro da janela desejada.

        Parâmetros:
            tk (str): nome do arquivo/ticker.

        Retorna:
            pd.Series: série temporal dos preços de fechamento ajustados no intervalo [ini:fim].
        """
        try:
            df = pd.read_csv(DIR / f"{tk}.csv", parse_dates=["Date"], index_col="Date")
            return df.loc[ini:fim, "Close"]
        except:
            return pd.Series(dtype="float64")  # Retorna série vazia se houver erro

    # Carrega os preços da empresa e do IBOVESPA
    s_emp = load(ticker)
    s_ibov = load(IBOV)

    # Verifica se houve erro na leitura
    if s_emp.empty or s_ibov.empty:
        print(f"⚠️ {emp} — dados ausentes\n")
        continue

    # ------------------------------ CÁLCULO DOS RETORNOS LOGARÍTMICOS ------------------------------

    r_emp = np.log(s_emp / s_emp.shift(1)).mul(100)     # Retorno percentual da empresa
    r_ibv = np.log(s_ibov / s_ibov.shift(1)).mul(100)   # Retorno percentual do IBOVESPA

    # Combina os retornos em um único DataFrame
    df = pd.concat([
        r_emp.rename(f"Retorno_{emp}"),
        r_ibv.rename("Retorno_Bovespa")
    ], axis=1).dropna()

    # ------------------------------ CRIAÇÃO DOS GRÁFICOS E SALVAMENTO ------------------------------

    # Cria subpasta para empresa e ano
    out_dir = ROOT / f"{emp} {ano}"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Histograma do retorno da empresa
    df[f"Retorno_{emp}"].plot(
        kind="hist", bins=30, alpha=0.65, edgecolor="black", figsize=(6,4)
    )
    plt.title(f"Histograma – {emp} ({ano})")
    plt.xlabel("Retorno diário (%)")
    plt.grid(True, ls="--", alpha=.3)
    plt.tight_layout()
    plt.savefig(out_dir / f"{emp}_{ano}_hist.png", dpi=300)
    plt.close()

    # Série temporal dos retornos
    df.plot(figsize=(10, 4))
    plt.title(f"{emp} vs Bovespa ({ano})")
    plt.ylabel("Retorno diário (%)")
    plt.grid(True, ls="--", alpha=.3)
    plt.tight_layout()
    plt.savefig(out_dir / f"{emp}_{ano}_linha.png", dpi=300)
    plt.close()

    # Estatísticas descritivas dos retornos (média, desvio, etc.)
    df.describe().to_csv(out_dir / f"{emp}_{ano}_stats.csv", float_format="%.2f")

print(f"\n✅ Gráficos e estatísticas organizados em subpastas dentro de: {ROOT.resolve()}")


✅ Gráficos e estatísticas organizados em subpastas dentro de: C:\Users\victo\OneDrive\Desktop\Projetos\Métodos Econométricos\Trabalho-Parte-1\Plot


## Tarefa 2: Apresente o diagrama de dispersão entre X e Y.

In [ ]:
# ------------------------------ LOOP PARA GERAR GRÁFICOS DE DISPERSÃO ------------------------------

for _, row in tabela.iterrows():
    emp = row["Empresas"]
    tick = TICKERS[emp]
    ref = datetime.strptime(row["Data de divulgação"], "%d/%m/%Y")  # Data de divulgação
    ini, fim = ref - timedelta(days=WINDOW), ref + timedelta(days=WINDOW)  # Janela de análise
    ano = ref.year

    # ------------------------------ FUNÇÃO PARA CARREGAR PREÇOS ------------------------------
    def load(tk):
        """
        Carrega os preços de fechamento ajustados de um ativo (ou índice) entre as datas 'ini' e 'fim'.

        Parâmetros:
            tk (str): ticker do ativo (ex: 'ITUB4.SA' ou '^BVSP').

        Retorna:
            pd.Series: série temporal dos preços de fechamento.
        """
        try:
            df = pd.read_csv(DIR / f"{tk}.csv", parse_dates=["Date"], index_col="Date")
            return df.loc[ini:fim, "Close"]
        except:
            return pd.Series(dtype="float64")  # Retorna série vazia se não conseguir carregar

    # Carrega os preços da empresa e do IBOV
    se = load(tick)  # Série da empresa
    si = load(IBOV)  # Série do índice IBOVESPA

    # Se qualquer uma das séries estiver vazia, exibe aviso e pula para a próxima
    if se.empty or si.empty:
        print(f"⚠️ Dados ausentes para {emp} — pulando dispersão.")
        continue

    # ------------------------------ CÁLCULO DOS RETORNOS DIÁRIOS ------------------------------

    re = np.log(se / se.shift(1)).mul(100)  # Retorno logarítmico da empresa (%)
    ri = np.log(si / si.shift(1)).mul(100)  # Retorno logarítmico do IBOV (%)

    # Junta os dois retornos em um DataFrame
    df = pd.concat([
        re.rename(f"Retorno_{emp}"),
        ri.rename("Retorno_Bovespa")
    ], axis=1).dropna()

    # ------------------------------ PLOTAGEM DO GRÁFICO DE DISPERSÃO ------------------------------

    out = PLOT_DIR / f"{emp} {ano}"         # Subpasta de saída
    out.mkdir(parents=True, exist_ok=True)  # Cria se não existir

    plt.figure(figsize=(12, 6))
    plt.scatter(
        df["Retorno_Bovespa"],
        df[f"Retorno_{emp}"],
        alpha=0.6,
        edgecolors='k'
    )
    plt.title(f'Dispersão dos Retornos – {emp} ({ano})')
    plt.xlabel("Retorno Bovespa")
    plt.ylabel(f"Retorno {emp}")
    plt.grid(True)
    plt.axhline(0, color='red')  # Linha horizontal no zero
    plt.axvline(0, color='red')  # Linha vertical no zero
    plt.tight_layout()
    plt.savefig(out / f"{emp}_{ano}_dispersao.png", dpi=300)
    plt.close()

print(f"\n✅ Gráficos de dispersão salvos em subpastas de: {PLOT_DIR.resolve()}")


✅ Gráficos de dispersão salvos em subpastas de: C:\Users\victo\OneDrive\Desktop\Projetos\Métodos Econométricos\Trabalho-Parte-1\Plot


### Tarefa 3: Estime o modelo linear $y = \alpha + \beta x + u$ e apresente os resultados na saída padrão do _Statsmodels_ no Python

In [26]:
import statsmodels.api as sm

# ------------------------------ LISTA PARA ARMAZENAR R² ------------------------------
r2 = []

# ------------------------------ LOOP SOBRE AS EMPRESAS ------------------------------
for _, row in tabela.iterrows():
    emp = row["Empresas"]
    tick = TICKERS[row["Empresas"]]
    ref = datetime.strptime(row["Data de divulgação"], "%d/%m/%Y")
    ano = row["Data de divulgação"][-4:]  # Extrai o ano como string
    ini, fim = ref - timedelta(days=WINDOW), ref + timedelta(days=WINDOW)

    # ------------------------------ FUNÇÃO PARA CARREGAR PREÇOS ------------------------------
    def load(tk):
        """
        Carrega os preços de fechamento ajustados de um ativo entre as datas 'ini' e 'fim'.

        Parâmetros:
            tk (str): ticker do ativo (ex: 'BRKM5.SA', '^BVSP').

        Retorna:
            pd.Series: preços de fechamento ajustados.
        """
        try:
            df = pd.read_csv(DIR / f"{tk}.csv", parse_dates=["Date"], index_col="Date")
            return df.loc[ini:fim, "Close"]
        except:
            return pd.Series(dtype="float64")  # Retorna série vazia se falhar

    # Carrega os preços da empresa e do IBOV
    se, si = load(tick), load(IBOV)

    # Pula se qualquer uma das séries estiver vazia
    if se.empty or si.empty:
        print(f"⚠️ {emp} — dados ausentes")
        continue

    # ------------------------------ CÁLCULO DOS RETORNOS DIÁRIOS ------------------------------

    re = np.log(se / se.shift(1)).mul(100)  # Retorno da empresa (%)
    ri = np.log(si / si.shift(1)).mul(100)  # Retorno do IBOVESPA (%)

    # Junta os dados em um DataFrame
    df = pd.concat([
        re.rename(f"Retorno_{emp}"),
        ri.rename("Retorno_Bovespa")
    ], axis=1).dropna()

    # ------------------------------ REGRESSÃO LINEAR ------------------------------

    X = sm.add_constant(df["Retorno_Bovespa"])  # Adiciona constante (intercepto)
    y = df[f"Retorno_{emp}"]                   # Variável dependente
    model = sm.OLS(y, X).fit()                 # Ajusta modelo OLS

    # Armazena nome da empresa, ano e R² arredondado
    r2.append((emp, ano, round(model.rsquared, 4)))

    # ------------------------------ SALVA O RESULTADO EM TXT ------------------------------
    out = PLOT_DIR / f"{emp} {ano}"
    out.mkdir(parents=True, exist_ok=True)

    with open(out / f"{emp}_{ano}_regressao.txt", "w", encoding="utf-8") as f:
        f.write(model.summary().as_text())

# ------------------------------ TABELA RESUMO DE R² ------------------------------

# Cria DataFrame com os resultados R²
df_r2 = pd.DataFrame(r2, columns=["Empresa", "Ano", "R²"]).sort_values(by=["Ano", "Empresa"])

# Cria figura com tabela de R²
fig, ax = plt.subplots(figsize=(8, len(df_r2)*0.5 + 1))
ax.axis("off")  # Oculta eixos

# Tabela com os dados
t = ax.table(
    cellText=df_r2.values,
    colLabels=df_r2.columns,
    cellLoc='center',
    loc='center'
)
t.auto_set_font_size(False)
t.set_fontsize(10)
t.scale(1.2, 1.2)

# Salva como imagem
plt.savefig(PLOT_DIR / "R-sqr.png", dpi=300, bbox_inches="tight")
plt.close()

print("\n✅ Resumos salvos por empresa\n✅ Tabela R² salva: Plot/R-sqr.png")


✅ Resumos salvos por empresa
✅ Tabela R² salva: Plot/R-sqr.png
